# Stage 07-D: Baselines, Ablations & Statistical Comparison

**Purpose:** Aggregate results from all baseline and ablation runs, produce the comparison table
from §5 of the research plan, run paired bootstrap significance testing (§6), and render the
final verdict on whether Path B (re-instantiated Vietnamese COOLANT) justifies its added
architectural complexity.

## Experiments compared

| ID | Notebook / Source | Modalities | Purpose |
|---|---|---|---|
| `baseline_gemma` | Published (AAAI 2025) | Text | Prior SOTA anchor (89.90% macro-F1) |
| `baseline_phobert_text` | **`03.9_vifactcheck_training`** (already trained) | Text | Text-only PhoBERT — strongest text baseline |
| `baseline_image_only` | Run separately | Image | Linear probe on image features |
| `baseline_naive_concat` | Run separately | Text+Image | PhoBERT CLS ⊕ ResNet mean, single FC |
| `06d_finetune` | `06d_finetune_fusion` | Text+Image | Prior best fusion (reference point) |
| `07c_full` | `07c_vietnamese_coolant_finetune` | Text+Image | Full Path B (main contribution) |
| `abl_no_itm` | Run separately (no consistency loss) | Text+Image | Ablation: w/o ITM |
| `abl_no_itc` | Run separately (no contrastive loss) | Text+Image | Ablation: w/o ITC |
| `abl_no_cmf` | Run separately (plain concat, no CMF) | Text+Image | Ablation: w/o Cross-Modal Fusion |
| `abl_no_att` | Run separately (uniform avg, no SE-Net) | Text+Image | Ablation: w/o Attention Aggregation |
| `abl_no_agu` | Run separately (no VAE/KL guidance) | Text+Image | Ablation: w/o Ambiguity Guidance |

> **Note on `baseline_phobert_text`**: `03.9_vifactcheck_training` already provides this —
> it saves a `checkpoint_manifest.json` in `training/checkpoints_vifactcheck/<run>/`.
> The loader below reads the most recent run. If `03.9` was trained with binary labels
> (`nei_as_real`) rather than `three_class`, the F1 figures will **not** be directly
> comparable to 07c (3-class). Verify `CONFIG['data']['label_variant']` in 03.9 before
> reading this number as a valid 3-class baseline.

## Significance testing

Paired bootstrap (10,000 iterations) on the test set, comparing Full 07c vs each baseline.
McNemar's test for binary correct/incorrect agreement between 07c and the strongest baseline.

## Outputs

- `training/stage07d_results/comparison_table.csv`
- `training/stage07d_results/comparison_bar_f1.png`
- `training/stage07d_results/ablation_bar_f1.png`
- `training/stage07d_results/confusion_matrices.png`
- `training/stage07d_results/bootstrap_significance.json`
- `training/stage07d_results/final_verdict.json`

In [ ]:
# ─── Environment Setup (do not edit) ─────────────────────────────────────────
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401
        return 'colab', False
    except ImportError:
        pass
    if Path('/workspace').exists() and os.environ.get('VAST_CONTAINERLABEL'):
        return 'vastai', False
    if Path('/workspace').exists():
        return 'vastai', True
    if sys.platform == 'win32': return 'windows', False
    if sys.platform == 'darwin': return 'mac', False
    return None, True


PLATFORM, _uncertain = _detect_platform()
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

PROJECT_ROOT = (
    Path('/content/drive/MyDrive/Thesis_Final/fake-news-detection') if PLATFORM == 'colab'
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    'colab':   PROJECT_ROOT / '.env.colab',
    'vastai':  PROJECT_ROOT / '.env.vastai',
    'windows': PROJECT_ROOT / '.env.windows',
    'mac':     PROJECT_ROOT / '.env.mac',
}
_env_file = _env_map.get(PLATFORM, PROJECT_ROOT / '.env')
if not _env_file.exists():
    _env_file = PROJECT_ROOT / '.env'

from dotenv import load_dotenv
load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()
print(f'Platform : {PLATFORM}  DATA_ROOT: {DATA_ROOT}')

In [ ]:
# ── Step 1: Configuration — Experiment Registry ───────────────────────────────
# For each experiment, provide one of:
#   results_file: path to a *_results.json  (standard format, 'test_metrics' key)
#   manifest_dir: path to a checkpoints directory that contains checkpoint_manifest.json
#                 files produced by 03.9_vifactcheck_training (most recent run is used)
#   manual: dict of test_metrics values (for published baselines without a file)

EXPERIMENTS = [
    # ── Published baseline (no file — reported from paper) ───────────────────
    {
        'id':       'baseline_gemma',
        'label':    'Gemma (AAAI 2025)',
        'modality': 'Text',
        'group':    'baseline',
        'color':    '#6B7280',
        'manual': {
            'test_macro_f1':  0.8990,   # [VERIFY] from AAAI 2025 paper
            'test_accuracy':  None,
            'test_f1_real':   None,
            'test_f1_fake':   None,
            'test_f1_nei':    None,
        },
    },
    # ── Text-only baseline: 03.9_vifactcheck_training (already trained) ──────
    # Loads from the most recent run in training/checkpoints_vifactcheck/.
    # IMPORTANT: verify 03.9 used label_variant='three_class' (not 'nei_as_real' binary)
    # before comparing its macro-F1 to the 3-class models below.
    {
        'id':          'baseline_phobert_text',
        'label':       'PhoBERT Text-Only (03.9)',
        'modality':    'Text',
        'group':       'baseline',
        'color':       '#0EA5E9',
        'manifest_dir': DATA_ROOT / 'training' / 'checkpoints_vifactcheck',
    },
    # ── Image-only baseline ───────────────────────────────────────────────────
    {
        'id':          'baseline_image_only',
        'label':       'Image Linear Probe',
        'modality':    'Image',
        'group':       'baseline',
        'color':       '#F59E0B',
        'results_file': DATA_ROOT / 'training' / 'stage07_baselines' / 'image_probe_results.json',
    },
    # ── Naive concat baseline ─────────────────────────────────────────────────
    {
        'id':          'baseline_naive_concat',
        'label':       'Naive Concat',
        'modality':    'Text+Image',
        'group':       'baseline',
        'color':       '#8B5CF6',
        'results_file': DATA_ROOT / 'training' / 'stage07_baselines' / 'naive_concat_results.json',
    },
    # ── Prior best fusion (reference from stage 06d) ──────────────────────────
    {
        'id':          '06d_finetune',
        'label':       '06d Fine-Tune Fusion',
        'modality':    'Text+Image',
        'group':       'baseline',
        'color':       '#EC4899',
        'results_file': DATA_ROOT / 'training' / 'stage06d_results',  # globs latest *_results.json
    },
    # ── Main contribution (Path B) ────────────────────────────────────────────
    {
        'id':          '07c_full',
        'label':       '07c Full Vi-COOLANT',
        'modality':    'Text+Image',
        'group':       'main',
        'color':       '#10B981',
        'results_file': DATA_ROOT / 'training' / 'stage07c_results' / 'finetune_results.json',
    },
    # ── Ablations ─────────────────────────────────────────────────────────────
    {
        'id': 'abl_no_itm', 'label': 'w/o ITM',
        'modality': 'Text+Image', 'group': 'ablation', 'color': '#F87171',
        'results_file': DATA_ROOT / 'training' / 'stage07_ablations' / 'no_itm_results.json',
    },
    {
        'id': 'abl_no_itc', 'label': 'w/o ITC',
        'modality': 'Text+Image', 'group': 'ablation', 'color': '#FB923C',
        'results_file': DATA_ROOT / 'training' / 'stage07_ablations' / 'no_itc_results.json',
    },
    {
        'id': 'abl_no_cmf', 'label': 'w/o CMF',
        'modality': 'Text+Image', 'group': 'ablation', 'color': '#FBBF24',
        'results_file': DATA_ROOT / 'training' / 'stage07_ablations' / 'no_cmf_results.json',
    },
    {
        'id': 'abl_no_att', 'label': 'w/o ATT',
        'modality': 'Text+Image', 'group': 'ablation', 'color': '#34D399',
        'results_file': DATA_ROOT / 'training' / 'stage07_ablations' / 'no_att_results.json',
    },
    {
        'id': 'abl_no_agu', 'label': 'w/o AGU',
        'modality': 'Text+Image', 'group': 'ablation', 'color': '#60A5FA',
        'results_file': DATA_ROOT / 'training' / 'stage07_ablations' / 'no_agu_results.json',
    },
]

OUTPUT_DIR = DATA_ROOT / 'training' / 'stage07d_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')
print(f'Registered experiments: {len(EXPERIMENTS)}')

In [ ]:
# ── Step 2: Dependencies ──────────────────────────────────────────────────────
import json, importlib, random
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

random.seed(42)
np.random.seed(42)
print('Dependencies loaded')

In [ ]:
# ── Step 3: Load Results ──────────────────────────────────────────────────────

def _load_results_file(p):
    """Load standard *_results.json. If p is a dir, globs for the most recent one."""
    p = Path(p)
    if p.is_dir():
        cands = sorted(p.glob('*_results.json'), key=lambda x: x.stat().st_mtime, reverse=True)
        if not cands: return None
        p = cands[0]
    if not p.exists(): return None
    with open(p) as f: return json.load(f)


def _load_manifest_dir(manifest_dir):
    """
    Load from 03.9_vifactcheck_training output.
    Globs training/checkpoints_vifactcheck/<run>/checkpoint_manifest.json,
    picks the most recent run, and maps its fields to the standard test_metrics shape.
    """
    d = Path(manifest_dir)
    if not d.exists():
        return None
    manifests = sorted(d.glob('*/checkpoint_manifest.json'),
                       key=lambda p: p.stat().st_mtime, reverse=True)
    if not manifests:
        return None
    with open(manifests[0]) as f:
        m = json.load(f)

    # checkpoint_manifest.json from 03.9 stores metrics under 'test_metrics' or
    # at top-level (depends on 03.9 version). Normalise to standard shape.
    tm_raw = m.get('test_metrics') or m.get('results', {}).get('test', {}) or {}

    # Field name mapping: 03.9 may use 'macro_f1' or 'test_macro_f1', etc.
    def _get(d, *keys, default=None):
        for k in keys:
            if k in d: return d[k]
        return default

    label_variant = (m.get('config', {}) or {}).get('data', {}).get('label_variant', 'unknown')
    num_classes   = (m.get('config', {}) or {}).get('data', {}).get('num_classes', '?')

    tm = {
        'test_macro_f1':  _get(tm_raw, 'macro_f1', 'test_macro_f1', 'val_macro_f1'),
        'test_accuracy':  _get(tm_raw, 'accuracy', 'test_accuracy', 'val_accuracy'),
        'test_f1_real':   _get(tm_raw, 'f1_real', 'test_f1_real', 'f1_class_0'),
        'test_f1_fake':   _get(tm_raw, 'f1_fake', 'test_f1_fake', 'f1_class_1'),
        'test_f1_nei':    _get(tm_raw, 'f1_nei',  'test_f1_nei',  'f1_class_2'),
        'confusion_matrix': _get(tm_raw, 'confusion_matrix'),
        'preds':          _get(tm_raw, 'preds', 'predictions'),
        'labels':         _get(tm_raw, 'labels', 'true_labels'),
        '_label_variant': label_variant,
        '_num_classes':   num_classes,
        '_manifest':      str(manifests[0]),
    }
    return {
        'test_metrics': tm,
        'best_epoch':   m.get('best_epoch') or m.get('epoch'),
        '_source':      str(manifests[0]),
    }


loaded = []
for exp in EXPERIMENTS:
    if 'manual' in exp:
        exp['results'] = {'test_metrics': exp['manual']}
        print(f'  [MANUAL]   {exp["label"]:35s} — macro_f1={exp["manual"]["test_macro_f1"]}')
        loaded.append(exp)

    elif 'manifest_dir' in exp:
        r = _load_manifest_dir(exp['manifest_dir'])
        if r is None:
            print(f'  [SKIP]     {exp["label"]:35s} — no manifest found in {exp["manifest_dir"]}')
        else:
            exp['results'] = r
            tm = r['test_metrics']
            f1 = tm.get('test_macro_f1') or 0
            variant = tm.get('_label_variant', '?')
            nc      = tm.get('_num_classes', '?')
            warn = ' [WARN: binary labels — not comparable to 3-class]' if variant == 'nei_as_real' else ''
            print(f'  [OK-03.9]  {exp["label"]:35s} — macro_f1={f1:.4f}  '
                  f'label_variant={variant} num_classes={nc}{warn}')
            print(f'             source: {tm["_manifest"]}')
            loaded.append(exp)

    elif 'results_file' in exp:
        r = _load_results_file(exp['results_file'])
        if r is None:
            print(f'  [SKIP]     {exp["label"]:35s} — not found ({exp["results_file"]})')
        else:
            exp['results'] = r
            f1 = r.get('test_metrics', {}).get('test_macro_f1', 0) or 0
            print(f'  [OK]       {exp["label"]:35s} — macro_f1={f1:.4f}')
            loaded.append(exp)

print(f'\nLoaded {len(loaded)}/{len(EXPERIMENTS)} experiments')

In [ ]:
# ── Step 4: Comparison Table ──────────────────────────────────────────────────
rows = []
for exp in loaded:
    tm = exp['results'].get('test_metrics', {})
    rows.append({
        'ID':            exp['id'],
        'Model':         exp['label'],
        'Modality':      exp['modality'],
        'Group':         exp['group'],
        'Macro-F1':      round(tm.get('test_macro_f1', 0) or 0, 4),
        'Accuracy':      round(tm.get('test_accuracy', 0) or 0, 4),
        'F1-Real':       round(tm.get('test_f1_real', 0) or 0, 4),
        'F1-Fake':       round(tm.get('test_f1_fake', 0) or 0, 4),
        'F1-NEI':        round(tm.get('test_f1_nei', 0) or 0, 4),
        'Best Epoch':    exp['results'].get('best_epoch', '-'),
    })

if not rows:
    print('No results loaded. Run the training notebooks first.')
else:
    df = pd.DataFrame(rows).sort_values('Macro-F1', ascending=False).reset_index(drop=True)
    df.index += 1
    df.index.name = 'Rank'

    print('\nFull Comparison Table (sorted by Test Macro-F1):')
    print('=' * 110)
    print(df.to_string())
    print('=' * 110)

    csv_p = OUTPUT_DIR / 'comparison_table.csv'
    df.to_csv(csv_p)
    print(f'\nTable saved: {csv_p}')

    best = df.iloc[0]
    main = next((r for r in rows if r['ID'] == '07c_full'), None)
    print(f'\n★  Best overall: [{best["Model"]}]  Macro-F1={best["Macro-F1"]:.4f}')
    if main:
        rank = df.index[df['ID'] == '07c_full'].tolist()
        print(f'   Path B (07c) rank: #{rank[0] if rank else "?"} with Macro-F1={main["Macro-F1"]:.4f}')

In [ ]:
# ── Step 5: Bar Charts — Baselines vs Main vs Ablations ──────────────────────
if not rows: raise RuntimeError('No results. Cannot plot.')

groups = ['baseline', 'main', 'ablation']
group_exps = {g: [e for e in loaded if e['group'] == g] for g in groups}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, subset_key, title in [
    (axes[0], ['baseline', 'main'], 'Baselines vs Full Path B'),
    (axes[1], ['main', 'ablation'], 'Ablations vs Full Path B'),
]:
    subset = [e for e in loaded if e['group'] in subset_key]
    if not subset:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue
    labels  = [e['label'] for e in subset]
    f1s     = [e['results'].get('test_metrics', {}).get('test_macro_f1', 0) or 0 for e in subset]
    colors  = [e['color'] for e in subset]
    bars = ax.bar(range(len(labels)), f1s, color=colors, edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, f1s):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    # Highlight Full Path B
    main_f1 = next((e['results'].get('test_metrics', {}).get('test_macro_f1', 0) or 0
                    for e in loaded if e['id'] == '07c_full'), None)
    if main_f1:
        ax.axhline(main_f1, color='#10B981', linestyle='--', linewidth=1.5,
                   alpha=0.8, label=f'Full Vi-COOLANT ({main_f1:.4f})')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Test Macro-F1'); ax.set_title(title)
    if main_f1 and f1s:
        ax.set_ylim(max(0, min(f1s) - 0.03), min(1.0, max(f1s) + 0.05))
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Stage 07-D: Comparison Summary', fontsize=13)
plt.tight_layout()
p = OUTPUT_DIR / 'comparison_bar_f1.png'
plt.savefig(p, dpi=150); plt.show(); plt.close()
print(f'Saved: {p}')

In [ ]:
# ── Step 6: Ablation Component Analysis ──────────────────────────────────────
ablations = [e for e in loaded if e['group'] == 'ablation']
main_exp  = next((e for e in loaded if e['id'] == '07c_full'), None)

if ablations and main_exp:
    main_f1 = main_exp['results'].get('test_metrics', {}).get('test_macro_f1', 0) or 0
    abl_labels = [e['label'] for e in ablations]
    abl_f1s    = [e['results'].get('test_metrics', {}).get('test_macro_f1', 0) or 0 for e in ablations]
    abl_deltas = [f1 - main_f1 for f1 in abl_f1s]  # negative = removing that component hurt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Absolute F1 bars with full model line
    colors = ['#22C55E' if d >= 0 else '#EF4444' for d in abl_deltas]
    axes[0].bar(range(len(abl_labels)), abl_f1s, color=colors, edgecolor='white')
    axes[0].axhline(main_f1, color='#10B981', linestyle='--', linewidth=2,
                    label=f'Full model ({main_f1:.4f})')
    for i, (val, delta) in enumerate(zip(abl_f1s, abl_deltas)):
        axes[0].text(i, val + 0.002, f'{val:.4f}\n({delta:+.4f})',
                     ha='center', va='bottom', fontsize=8)
    axes[0].set_xticks(range(len(abl_labels)))
    axes[0].set_xticklabels(abl_labels, rotation=15, ha='right', fontsize=9)
    axes[0].set_ylabel('Test Macro-F1')
    axes[0].set_title('Ablation F1 (green = better, red = worse than full model)')
    axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)

    # Delta bars
    delta_colors = ['#22C55E' if d >= 0 else '#EF4444' for d in abl_deltas]
    axes[1].bar(range(len(abl_labels)), abl_deltas, color=delta_colors, edgecolor='white')
    axes[1].axhline(0, color='black', linewidth=1.2)
    for i, d in enumerate(abl_deltas):
        axes[1].text(i, d + 0.001 if d >= 0 else d - 0.004,
                     f'{d:+.4f}', ha='center', va='bottom' if d >= 0 else 'top', fontsize=9)
    axes[1].set_xticks(range(len(abl_labels)))
    axes[1].set_xticklabels(abl_labels, rotation=15, ha='right', fontsize=9)
    axes[1].set_ylabel('ΔMacro-F1 vs Full Model')
    axes[1].set_title('Component Contribution (negative = component helps)')
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle('COOLANT Component Ablation Analysis', fontsize=13)
    plt.tight_layout()
    p = OUTPUT_DIR / 'ablation_bar_f1.png'
    plt.savefig(p, dpi=150); plt.show(); plt.close()
    print(f'Saved: {p}')

    # Identify most/least impactful components
    sorted_abl = sorted(zip(abl_labels, abl_deltas), key=lambda x: x[1])
    print('\nComponent importance (most negative delta = most important component):')
    for label, d in sorted_abl:
        print(f'  {label:15s}: ΔF1={d:+.4f}')
else:
    print('Ablation results not yet available — run ablation notebooks first.')

In [ ]:
# ── Step 7: Paired Bootstrap Significance Testing ─────────────────────────────
# 10,000 iterations, paired test: resampled Macro-F1 delta between Full 07c and each baseline.
# Requires preds and labels from each experiment's results JSON.

N_BOOTSTRAP = 10_000


def macro_f1_from_arrays(labels, preds, n_classes=3):
    per_class = []
    for c in range(n_classes):
        tp = sum(l == c and p == c for l, p in zip(labels, preds))
        fp = sum(l != c and p == c for l, p in zip(labels, preds))
        fn = sum(l == c and p != c for l, p in zip(labels, preds))
        if tp + fp + fn == 0:
            per_class.append(0.0)
        else:
            p_  = tp / (tp + fp) if (tp + fp) > 0 else 0
            r_  = tp / (tp + fn) if (tp + fn) > 0 else 0
            per_class.append(2 * p_ * r_ / (p_ + r_) if (p_ + r_) > 0 else 0.0)
    return np.mean(per_class)


def paired_bootstrap(labels, preds_a, preds_b, n_iter=N_BOOTSTRAP, seed=42):
    """Returns (delta_observed, ci_lower, ci_upper, p_value)."""
    rng     = np.random.default_rng(seed)
    n       = len(labels)
    obs_a   = macro_f1_from_arrays(labels, preds_a)
    obs_b   = macro_f1_from_arrays(labels, preds_b)
    obs_delta = obs_b - obs_a  # positive = b is better
    deltas  = np.zeros(n_iter)
    for i in range(n_iter):
        idx     = rng.integers(0, n, size=n)
        s_lbl   = [labels[j]   for j in idx]
        s_pa    = [preds_a[j]  for j in idx]
        s_pb    = [preds_b[j]  for j in idx]
        deltas[i] = macro_f1_from_arrays(s_lbl, s_pb) - macro_f1_from_arrays(s_lbl, s_pa)
    ci_lo  = float(np.percentile(deltas, 2.5))
    ci_hi  = float(np.percentile(deltas, 97.5))
    p_val  = float(np.mean(deltas <= 0))  # one-sided p: how often is delta <= 0?
    return float(obs_delta), ci_lo, ci_hi, p_val


# Find 07c preds/labels
main_exp = next((e for e in loaded if e['id'] == '07c_full'), None)
sig_results = []

if main_exp and main_exp['results'].get('test_metrics', {}).get('preds'):
    labels_main = main_exp['results']['test_metrics']['labels']
    preds_main  = main_exp['results']['test_metrics']['preds']

    compare_against = [e for e in loaded
                       if e['id'] != '07c_full'
                       and e.get('results', {}).get('test_metrics', {}).get('preds')]

    for exp in compare_against:
        labels_b = exp['results']['test_metrics']['labels']
        preds_b  = exp['results']['test_metrics']['preds']
        if len(labels_b) != len(labels_main):
            print(f'[SKIP] {exp["label"]}: test set size mismatch')
            continue

        delta, ci_lo, ci_hi, p_val = paired_bootstrap(labels_main, preds_b, preds_main)
        sig = '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        sig_results.append({
            'vs':       exp['label'],
            'delta':    round(delta, 4),
            'ci_95':    [round(ci_lo, 4), round(ci_hi, 4)],
            'p_value':  round(p_val, 4),
            'sig':      sig,
        })
        print(f'07c vs {exp["label"]:30s}: ΔF1={delta:+.4f} 95%CI=[{ci_lo:.4f},{ci_hi:.4f}] p={p_val:.4f} {sig}')

    if sig_results:
        sp = OUTPUT_DIR / 'bootstrap_significance.json'
        with open(sp, 'w') as f:
            json.dump({'n_bootstrap': N_BOOTSTRAP, 'results': sig_results}, f, indent=2)
        print(f'\nSignificance results saved: {sp}')
else:
    print('Bootstrap skipped: 07c preds/labels not found in results JSON.')
    print('  Ensure 07c saves preds and labels in test_metrics.')

In [ ]:
# ── Step 8: Confusion Matrices (Key Models) ───────────────────────────────────
key_ids = ['07c_full', 'baseline_phobert_text', 'baseline_naive_concat']
cms = [
    (e['label'], e['results'].get('test_metrics', {}).get('confusion_matrix'))
    for e in loaded
    if e['id'] in key_ids and e['results'].get('test_metrics', {}).get('confusion_matrix')
]

if cms:
    n = len(cms)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 5))
    if n == 1: axes = [axes]
    cmaps = ['Greens', 'Blues', 'Purples']
    for (label, cm), ax, cmap in zip(cms, axes, cmaps * 2):
        cm_arr = np.array(cm)
        sns.heatmap(
            cm_arr, annot=True, fmt='d', cmap=cmap,
            xticklabels=['Real', 'Fake', 'NEI'],
            yticklabels=['Real', 'Fake', 'NEI'],
            ax=ax, cbar=False,
        )
        ax.set_title(label, fontsize=10)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.suptitle('Confusion Matrices — Key Models', fontsize=12, y=1.02)
    plt.tight_layout()
    p = OUTPUT_DIR / 'confusion_matrices.png'
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
    print(f'Saved: {p}')
else:
    print('No confusion matrices available for key models yet.')

In [ ]:
# ── Step 9: Final Verdict ─────────────────────────────────────────────────────
from datetime import datetime

main_exp = next((e for e in loaded if e['id'] == '07c_full'), None)
baseline_text = next((e for e in loaded if e['id'] == 'baseline_phobert_text'), None)
gemma = next((e for e in loaded if e['id'] == 'baseline_gemma'), None)

print('=' * 70)
print('STAGE 07-D — FINAL VERDICT')
print('=' * 70)

if not loaded:
    print('No results loaded. Run training notebooks first.')
else:
    sorted_exp = sorted(loaded, key=lambda e: e['results'].get('test_metrics', {}).get('test_macro_f1', 0) or 0, reverse=True)
    for rank, exp in enumerate(sorted_exp, 1):
        tm = exp['results'].get('test_metrics', {})
        f1  = tm.get('test_macro_f1', 0) or 0
        acc = tm.get('test_accuracy', 0) or 0
        nei = tm.get('test_f1_nei', 0) or 0
        marker = '★ ' if rank == 1 else f'{rank}. '
        print(f'{marker}[{exp["label"]}]  F1={f1:.4f}  Acc={acc:.4f}  F1-NEI={nei:.4f}')

    print()
    if main_exp:
        main_f1 = main_exp['results'].get('test_metrics', {}).get('test_macro_f1', 0) or 0

        if baseline_text:
            text_f1 = baseline_text['results'].get('test_metrics', {}).get('test_macro_f1', 0) or 0
            delta_vs_text = main_f1 - text_f1
            print(f'Path B vs Text-Only PhoBERT: {delta_vs_text:+.4f}')
            sig_entry = next((s for s in sig_results if s['vs'] == baseline_text['label']), None)
            if sig_entry:
                print(f'  95% CI: [{sig_entry["ci_95"][0]:.4f}, {sig_entry["ci_95"][1]:.4f}]  '
                      f'p={sig_entry["p_value"]:.4f} {sig_entry["sig"]}')

        if gemma:
            gem_f1 = gemma['results'].get('test_metrics', {}).get('test_macro_f1', 0)
            print(f'Path B vs Gemma SOTA anchor : {main_f1 - gem_f1:+.4f}')

        print()
        if main_f1 > (text_f1 if baseline_text else 0):
            print('[CONCLUSION] Multimodal Path B improves over text-only baseline.')
            print('  Image evidence contributes signal to ViFactCheck-MM.')
        else:
            print('[CONCLUSION] Path B does NOT significantly outperform text-only baseline.')
            print('  Report this as a negative/inconclusive result (still a valid contribution).')
            print('  Recommended: report Path A (frozen COOLANT + text) as practical recommendation.')

        # NEI analysis
        print('\n--- NEI (hard class) F1 across models ---')
        for exp in sorted_exp:
            nei = exp['results'].get('test_metrics', {}).get('test_f1_nei', 0) or 0
            print(f'  {exp["label"]:35s}: F1-NEI={nei:.4f}')

verdict = {
    'generated':        datetime.now().isoformat(),
    'n_experiments':    len(loaded),
    'ranking': [
        {
            'rank':  i+1,
            'id':    exp['id'],
            'label': exp['label'],
            'test_macro_f1': exp['results'].get('test_metrics', {}).get('test_macro_f1'),
            'test_accuracy': exp['results'].get('test_metrics', {}).get('test_accuracy'),
            'test_f1_nei':   exp['results'].get('test_metrics', {}).get('test_f1_nei'),
        }
        for i, exp in enumerate(sorted_exp)
    ],
    'bootstrap_tests': sig_results,
}
vp = OUTPUT_DIR / 'final_verdict.json'
with open(vp, 'w') as f: json.dump(verdict, f, indent=2)
print(f'\nVerdict saved: {vp}')
print('=' * 70)
print(f'Report: {datetime.now().strftime("%Y-%m-%d %H:%M")}')